# LifeStack GRPO — Training Evidence  (Run v4)

This notebook parses `train_run_v4.log` (committed to the repo) and generates  
publication-quality plots for the hackathon submission.  

**Run this on the HuggingFace Jupyter notebook server** (or Colab) to regenerate the plots.

### What you get
| Output file | Description |
|---|---|
| `plots/training_summary.png` | 4-panel: r₀, r_lt, component breakdown, loss |
| `plots/reward_curve.png` | Immediate reward with rolling mean + std band |
| `plots/reward_components.png` | task_success / milestone / replan / longterm sub-plots |
| `plots/loss_curve.png` | HF Trainer scalar loss (if present in log) |

In [ ]:
# Cell 1 – Locate the repo root automatically
import os, sys
from pathlib import Path

# Works whether you open the notebook from the repo root or the notebooks/ subfolder
cwd = Path.cwd()
if (cwd / 'scripts' / 'plot_training.py').exists():
    REPO = cwd
elif (cwd.parent / 'scripts' / 'plot_training.py').exists():
    REPO = cwd.parent
else:
    # HF server default: ~/app/Meta-R2
    REPO = Path(os.environ.get('HOME', '/root')) / 'app' / 'Meta-R2'

print(f'Repo root : {REPO}')
os.chdir(REPO)

LOG_FILE = REPO / 'train_run_v4.log'
OUT_DIR  = REPO / 'plots'

print(f'Log file  : {LOG_FILE}  (exists={LOG_FILE.exists()})')
print(f'Output dir: {OUT_DIR}')

In [ ]:
# Cell 2 – Install matplotlib if not present (usually already installed)
import importlib
if importlib.util.find_spec('matplotlib') is None:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'], check=True)

import matplotlib
print('matplotlib', matplotlib.__version__)
import numpy as np
print('numpy', np.__version__)

In [ ]:
# Cell 3 – Run plot_training.py to generate all PNG files
import subprocess, sys
from pathlib import Path

REPO     = Path.cwd()
LOG_FILE = REPO / 'train_run_v4.log'
OUT_DIR  = REPO / 'plots'

if not LOG_FILE.exists():
    print(f'ERROR: {LOG_FILE} not found!')
    print('Make sure you are running from the Meta-R2 repo root.')
else:
    result = subprocess.run(
        [
            sys.executable,
            str(REPO / 'scripts' / 'plot_training.py'),
            '--log', str(LOG_FILE),
            '--out', str(OUT_DIR),
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print('--- STDERR ---')
        print(result.stderr[-3000:])   # last 3 kB only
    if result.returncode == 0:
        print('\nAll plots generated successfully.')
    else:
        print(f'\nplot_training.py exited with code {result.returncode}')

In [ ]:
# Cell 4 – Display the 4-panel training summary
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

REPO = Path.cwd()
p    = REPO / 'plots' / 'training_summary.png'

if p.exists():
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(mpimg.imread(str(p)))
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'Not found: {p}  — run Cell 3 first.')

In [ ]:
# Cell 5 – Display reward + components side by side
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

REPO = Path.cwd()
plots = [
    ('Reward Curve',      REPO / 'plots' / 'reward_curve.png'),
    ('Reward Components', REPO / 'plots' / 'reward_components.png'),
]

for title, p in plots:
    if p.exists():
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(mpimg.imread(str(p)))
        ax.axis('off')
        ax.set_title(title, fontsize=11, pad=6)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Not found: {p}')

In [ ]:
# Cell 6 – Display loss curve (if HF Trainer emitted scalar loss)
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

REPO = Path.cwd()
p = REPO / 'plots' / 'loss_curve.png'
if p.exists():
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(mpimg.imread(str(p)))
    ax.axis('off')
    ax.set_title('Training Loss Curve', fontsize=11, pad=6)
    plt.tight_layout()
    plt.show()
else:
    print('loss_curve.png not found (TRL GRPO may not emit a scalar loss — reward curve is the primary evidence).')

In [ ]:
# Cell 7 – Print numeric training statistics from the log
import re, json
import numpy as np
from pathlib import Path

REPO     = Path.cwd()
LOG_FILE = REPO / 'train_run_v4.log'

if not LOG_FILE.exists():
    print('Log not found.')
else:
    text = LOG_FILE.read_text(errors='replace')
    file_size_mb = LOG_FILE.stat().st_size / 1e6

    # ── Parse reward lines ──────────────────────────────────────────────────
    STEP_RE = re.compile(
        r'\[step\s+(\d+)\]\s+r0=([+-]?\d+\.\d+).*?r_lt=([+-]?\d+\.\d+)'
    )
    matches  = STEP_RE.findall(text)
    n_steps  = len(matches)

    if n_steps > 0:
        steps   = np.array([int(m[0])   for m in matches])
        rewards = np.array([float(m[1]) for m in matches])
        ltr     = np.array([float(m[2]) for m in matches])
        q1  = max(1, n_steps // 10)
        q2  = n_steps - q1
        imp = np.mean(rewards[q2:]) - np.mean(rewards[:q1])

        SEP = '═' * 56
        print(SEP)
        print('  LifeStack GRPO  ─  Run v4  ─  Training Statistics')
        print(SEP)
        print(f'  Log size (MB)        : {file_size_mb:>10.2f}')
        print(f'  Reward steps logged  : {n_steps:>10,}')
        print(f'  Step range           : {steps[0]:>10,} → {steps[-1]:,}')
        print()
        print(f'  Immediate reward r₀')
        print(f'    Mean  (all steps)  : {np.mean(rewards):>10.4f}')
        print(f'    Mean  (first 10 %) : {np.mean(rewards[:q1]):>10.4f}')
        print(f'    Mean  (last  10 %) : {np.mean(rewards[q2:]):>10.4f}')
        print(f'    Net improvement Δ  : {imp:>+10.4f}  ← key metric')
        print(f'    Std dev            : {np.std(rewards):>10.4f}')
        print(f'    Min / Max          : {rewards.min():>10.4f} / {rewards.max():.4f}')
        print()
        print(f'  Long-term reward r_lt')
        print(f'    Mean  (all steps)  : {np.mean(ltr):>10.4f}')
        print(f'    Net improvement Δ  : {np.mean(ltr[q2:]) - np.mean(ltr[:q1]):>+10.4f}')
        print(SEP)
    else:
        print('No [step N] r0=... lines found in log.')
        print('Showing last 40 lines of log for debugging:')
        print('\n'.join(text.splitlines()[-40:]))

    # ── Parse HF Trainer dict losses ────────────────────────────────────────
    DICT_RE = re.compile(r'\{[^{}]+\}')
    losses = []
    for m in DICT_RE.finditer(text):
        try:
            d = eval(m.group(0), {'__builtins__': {}})
            if isinstance(d, dict) and 'loss' in d:
                losses.append(float(d['loss']))
        except Exception:
            pass

    if losses:
        lv = np.array(losses)
        q  = max(1, len(lv) // 10)
        print()
        print(f'  HF Trainer loss entries : {len(lv):>8,}')
        print(f'  Loss start (first 10 %) : {np.mean(lv[:q]):>8.4f}')
        print(f'  Loss end   (last  10 %) : {np.mean(lv[-q:]):>8.4f}')
        print(f'  Net Δ loss              : {np.mean(lv[-q:]) - np.mean(lv[:q]):>+8.4f}')
        print(SEP)

In [ ]:
# Cell 8 – Show the pre-committed grpo_reward_curve_v4.png (always available from repo)
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

REPO = Path.cwd()
p = REPO / 'grpo_reward_curve_v4.png'

if p.exists():
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(str(p)))
    ax.axis('off')
    ax.set_title('grpo_reward_curve_v4.png  (committed artefact from real run)', fontsize=11, pad=6)
    plt.tight_layout()
    plt.show()
else:
    print(f'{p} not found – make sure you are running from the repo root.')

In [ ]:
# Cell 9 (Optional) – Download plots as a zip to your local machine
import zipfile, os
from pathlib import Path

REPO    = Path.cwd()
OUT_DIR = REPO / 'plots'
ZIP_OUT = REPO / 'training_plots_v4.zip'

pngs = list(OUT_DIR.glob('*.png'))
if pngs:
    with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in pngs:
            zf.write(f, f.name)
        # Also include the pre-committed artefact
        extra = REPO / 'grpo_reward_curve_v4.png'
        if extra.exists():
            zf.write(extra, extra.name)
    print(f'Zip created: {ZIP_OUT}  ({ZIP_OUT.stat().st_size / 1024:.1f} KB)')
    print('Download it from the JupyterLab file browser on the left.')
else:
    print('No plots to zip yet. Run Cell 3 first.')